# GridSketcher: a conformal regional grid from a sketched outline

`GridSketcher` builds an orthogonal (conformal) regional `Grid` from a hand-drawn coastline outline and four chosen corners, instead of a bounding box or a projected extent. It is a good fit whenever the domain boundary should follow an irregular coastline rather than straight lon/lat or projected edges.

The same solve is also available without the widget, as `Grid.from_outline` -- see "Building a grid directly from an outline" below. Either way the result is a plain `Grid` that remembers the outline it came from, so the two are interchangeable: open `GridSketcher(grid)` on a grid built by `Grid.from_outline` to keep refining it, or call `Grid.from_outline` once you are happy with a sketch built interactively.

This notebook is a short walkthrough of the widget. The widgets on the docs page were saved when the docs were built: they show the map and the grid, but clicks do nothing there without a running Python. Run the notebook yourself to sketch. The map loads OpenLayers from `cdn.jsdelivr.net`, so the browser needs internet access.

## 1. Launch the sketcher

With no arguments, `GridSketcher` starts from a small default box (the CrocoDash tutorial domain). `resolution_km` sets the nominal (mean) cell size.

In [1]:
from mom6_forge.widgets import GridSketcher

sketch = GridSketcher(resolution_km=10)
sketch

/home/runner/micromamba/envs/mom6_forge/lib/python3.13/site-packages/cartopy/io/__init__.py:263: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/50m_physical/ne_50m_land.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)


## 2. Sketch the outline

The map is drawn in the projection the grid is solved in, so an edge drawn straight on the screen is straight for the solver. On the map, as the line under it also says:

- **Drag** a vertex to move it.
- **Drag an edge** to insert a new vertex there.
- **Right-click** (or **Alt-click**) a vertex to delete it -- the remaining vertices keep their corner status.
- **Double-click** a vertex to make or unmake it one of the outline's 4 corners (numbered 1-4 on the map).
- **Draw box**, then drag on the map, for a new 4-corner box.
- **Scroll** to zoom and drag the map to pan; **Zoom to outline** shows the whole outline again.
- **Clear** to draw a new outline: click points, then click point 1 again to close it (4 points become the 4 corners).
- Grid lines turn orange or red where ocean cells are badly shaped (orthogonality, size ratio or aspect ratio). Only ocean cells are judged (from the coastline).

On the panel: **Undo**/**Redo**, the **Projection** buttons (the projection the grid is solved in, and the map is drawn in; the pressed one is in use; the green one is recommended, as its cells vary least in size, and a new sketch or newly drawn outline is solved on it until you press another), **Resolution** (km), and the diagnostics shown in **Preview** and **Details**. **Errors** say why **Build grid** is greyed out (fewer than 4 corners, crossing edges, a failed solve, folded ocean cells) and **Warnings** list what to check (abrupt size changes, non-orthogonal or stretched cells, nearly flat corners, short or coast-hugging open boundaries), one line each; any warning turns Build amber.

## 3. Build and save the grid

**Build grid** solves the outline at full resolution and stores the result as `sketch.grid` -- an ordinary `mom6_forge.grid.Grid`. Like `Grid.from_outline` below, it carries a `.outline` attribute, so `GridSketcher(sketch.grid)` reopens the same outline for further editing.

Nobody clicked **Build grid** when these docs were built, so the cell below builds the same grid in Python when `sketch.grid` is empty: `Grid.from_outline` with the sketch's outline, corners, resolution and projection is exactly what Build does.

In [2]:
from mom6_forge.grid import Grid

o = sketch.outline
grid = sketch.grid or Grid.from_outline(
    o.lon, o.lat, o.corners,
    resolution_km=sketch.resolution_km,
    projection=sketch.projection,
    name="sketch",
)
grid.nx, grid.ny

(73, 89)

**Save** writes the built grid to `GridLibrary/grid_<name>.nc`. The file keeps the outline, so `GridSketcher(Grid.from_supergrid_file(path))` reopens the sketch; after any edit, press **Build grid** again before saving. Tick **Overwrite** to replace a saved grid of the same name.

## 4. Use the grid like any other

`grid` is a regular `Grid`: pass it to `Topo` (and to CrocoDash's `Case`) exactly as with a grid built from `Grid.from_projection`, `Grid.from_center`, or `Grid.from_outline`. Once it has a bathymetry, `TopoEditor(topo)`'s **Show open boundaries** shows where its open boundaries are.

In [3]:
from mom6_forge.topo import Topo

topo = Topo(grid, min_depth=10.0)

## 5. Starting from an existing grid

`GridSketcher(grid)` starts a new sketch from an existing grid. If the grid has an `.outline` (built by `Grid.from_outline`, or by a previous "Build grid"), that outline is reopened exactly as it was. Any other grid has its own boundary traced, with `per_side` vertices per side (default 1, a plain 4-vertex outline); use more for a boundary that isn't a straight line in the sketching projection, such as a lon/lat box -- useful for re-sketching a domain someone already built.

In [4]:
box = Grid(lenx=8.0, leny=8.0, resolution=0.1, xstart=235.0, ystart=31.0, name="box")
resketch = GridSketcher(box, per_side=4)  # box has no .outline: its edge is traced
resketch

## 6. Building a grid directly from an outline

`Grid.from_outline` is `GridSketcher`'s solve without the widget: give it an outline (>= 4 vertices, in degrees) and the 4 corner indices that mark the grid's corners (defaulted to `[0, 1, 2, 3]` for a plain 4-vertex outline, required otherwise) and it returns a `Grid` directly -- handy in a script, or once a sketched outline's coordinates are already known.

Set either `resolution_km` (the nominal, mean cell size -- the same quantity the widget's **Resolution** control sets) or `n_cells` (target `nx * ny`), and `projection` (a kind name such as `"lcc"`, or `"auto"`, the default, which picks the widget's recommendation: the kind whose cells vary least in size).

In [5]:
lon = [235.0, 243.0, 243.0, 235.0]
lat = [31.0, 31.0, 39.0, 39.0]
direct_grid = Grid.from_outline(lon, lat, resolution_km=10, name="direct")
direct_grid.outline["projection"]

{'kind': 'merc', 'lon_0': -121.0, 'lat_0': 0.0, 'lat_1': None, 'lat_2': None}

`direct_grid.outline` records the outline, corners, resolved projection, and resolution it was built from, so it can still be opened in the widget to keep refining it interactively:

In [6]:
sketch2 = GridSketcher(direct_grid)  # reopens direct_grid.outline exactly
sketch2